In [1]:
import pandas as pd

df_answers = pd.read_csv("data/rag-answers-new.csv")
answers = df_answers.to_dict(orient="records")

In [2]:
from pydantic import BaseModel, Field
from typing import Literal

class AnswerEvaluation(BaseModel):
    reasoning: str = Field(
        description="Reasoning about the quality of the answer."
    )
    score: Literal["good", "bad"] = Field(
        description="'good' if the answer is correct and complete, 'bad' otherwise."
    )

In [3]:
aqa_judge_instructions = """
You are an expert evaluator. You will be given:
1. A question from a student
2. The original answer from the FAQ (ground truth)
3. An answer generated by an AI assistant

Your task is to decide if the AI answer is semantically equivalent to
the original answer.

Rules:
- The AI answer does NOT need to be word-for-word identical
- It should convey the same key information
- Extra detail is fine as long as the core answer is correct
- Mark 'bad' only if the AI answer is wrong or misses the key point

Be fair and focus on correctness, not style.
""".strip()

In [4]:
aqa_judge_prompt = """
Question:
{question}

Original Answer (ground truth):
{answer_orig}

AI Answer:
{answer_llm}
""".strip()

In [5]:
from dotenv import load_dotenv
from openai import OpenAI
from evaluation_utils import calc_price, calc_total_price, llm_structured_retry, map_progress

load_dotenv()
openai_client = OpenAI()

In [8]:
rec = answers[0]

In [9]:
prompt = aqa_judge_prompt.format(
    question=rec["question"],
    answer_orig=rec["answer_orig"],
    answer_llm=rec["answer_llm"]
)

In [10]:
eval_result, usage = llm_structured_retry(
    openai_client,
    aqa_judge_instructions,
    prompt,
    AnswerEvaluation,
)

eval_result

AnswerEvaluation(reasoning='The AI answer preserves the main point: late enrollment is allowed and certificate eligibility depends on submitting the project while submissions are open. It adds that certificates are only for the live cohort, which is extra detail not present in the ground truth but does not contradict it. Semantic equivalence is maintained.', score='good')

In [11]:
calc_price(usage)

{'input_cost': 0.0002325,
 'output_cost': 0.00033299999999999996,
 'total_cost': 0.0005655}

In [12]:
def evaluate_aqa(question, answer_orig, answer_llm, model="gpt-5.4-mini"):
    prompt = aqa_judge_prompt.format(
        question=question,
        answer_orig=answer_orig,
        answer_llm=answer_llm
    )

    result, usage = llm_structured_retry(
        openai_client,
        aqa_judge_instructions,
        prompt,
        AnswerEvaluation,
        model=model,
    )

    return result, usage

In [13]:
eval_result, usage = evaluate_aqa(
    question=rec["question"],
    answer_orig=rec["answer_orig"],
    answer_llm=rec["answer_llm"]
)

eval_result

AnswerEvaluation(reasoning='The AI answer preserves the key point: late enrollment is allowed, and certificate eligibility requires submitting the project while submissions are still open. It adds an extra detail about certificates not being available for self-paced mode, which is not in the ground truth but does not contradict the core answer. The semantic meaning matches the original.', score='good')

In [14]:
def judge_record(rec):
    eval_result, usage = evaluate_aqa(
        question=rec["question"],
        answer_orig=rec["answer_orig"],
        answer_llm=rec["answer_llm"]
    )

    result = {
        "question": rec["question"],
        "document": rec["document"],
        "score": eval_result.score,
        "reasoning": eval_result.reasoning,
    }

    return result, usage

In [15]:
answers[:10]

[{'question': 'I found this course late — can I still enroll and follow along?',
  'answer_llm': 'Yes, you can still join and follow along.\n\nIf you want a certificate, though, you must submit your project while the course is still accepting submissions, and certificates are only available for the live cohort—not self-paced mode.',
  'answer_orig': 'Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.',
  'document': '74eb249bbf'},
 {'question': 'Is it too late to join the course if I only discovered it now?',
  'answer_llm': 'Yes, you can still join the course if you just discovered it now. If you want a certificate, you need to submit your project while the course is still accepting submissions.',
  'answer_orig': 'Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.',
  'document': '74eb249bbf'},
 {'question': 'Can I start the course after it has alr

In [16]:
judge_record[:10]

TypeError: 'function' object is not subscriptable

In [17]:
from concurrent.futures import ThreadPoolExecutor

with ThreadPoolExecutor(max_workers=6) as pool:
    results = map_progress(pool, answers[:10], judge_record)

  0%|          | 0/10 [00:00<?, ?it/s]

In [18]:
evaluations = []
usages = []

for evaluation, usage in results:
    evaluations.append(evaluation)
    usages.append(usage)

In [19]:
df_eval = pd.DataFrame(evaluations)

In [20]:
calc_total_price(usages)

0.00572925

In [21]:
good_count = (df_eval["score"] == "good").sum()
total_count = len(df_eval)
print(f"Good: {good_count}/{total_count} = {good_count/total_count:.2%}")

Good: 9/10 = 90.00%


In [23]:
df_eval[df_eval["score"] == "bad"].head().values

array([['Can I start the course after it has already begun?',
        '74eb249bbf', 'bad',
        'The ground truth says you can start after the course has begun, but to receive a certificate you must submit your project while submissions are still accepted. The AI answer correctly says you can start whenever you want, but it omits the key certificate/submission condition and instead adds unrelated details about deadlines. Since the certificate requirement is the important part of the original answer, this is incomplete.']],
      dtype=object)

In [24]:
df_eval.to_csv("data/rag-evaluations-new_partial.csv", index=False)